<a href="https://colab.research.google.com/github/RAseng77/computer_vision_study/blob/main/study-notes/dinov2__practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow faiss-cpu supervision -q

In [3]:
import roboflow
import os

# roboflow에 로그인합니다.
roboflow.login()

# roboflow 객체 생성
rf = roboflow.Roboflow()

# "team-roboflow" 워크스페이스에있는 "coco-128" 프로젝트를 가져옵니다.
project = rf.workspace("team-roboflow").project("coco-128")

# 프로젝트의 두 번째 버전을 선택하고, "coco" 포맷으로 다운로드 합니다.
dataset = project.version(2).download("coco")



visit https://app.roboflow.com/auth-cli to get your authentication token.
Paste the authentication token here: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to COCO-128-2 in coco:: 100%|██████████| 386/386 [00:00<00:00, 4065.26it/s]


In [4]:
# 현재 작업 디렉토리의 경로를 가져옵니다.
cwd = os.getcwd()

# 현재 작업 디렉토리 경로에 "COCO-128-2/train/" 경로를 결합합니다.
ROOT_DIR = os.path.join(cwd, "COCO-128-2/train/")

# 'ROOT_DIR'에 있는 모든 파일과 하위 디렉토리의 이름을 리스트로 가져옵니다.
files = os.listdir(ROOT_DIR)

files = [os.path.join(ROOT_DIR, f) for f in files if f.lower().endswith(".jpg")]

In [5]:
import faiss
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import cv2
import json
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
import supervision as sv

In [6]:
# Pytorch Hub를 사용하여 'facebookresearch/dinov2' 저장소에서 'dinov2_vits14'모델을 불러옵니다.
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dinov2_vits14.to(device)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 350MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [8]:
transform_image = T.Compose([
    T.ToTensor(), # 이미지를 PyTorch 텐서로 변환합니다.
    T.Resize(224), # 이미지의 가장 짧은 변을 224 픽셀로 재조정합니다.
    T.CenterCrop(224), # 이미지 중앙을 기준으로 224x224 픽셀 크기로 잘라냅니다.
    T.Normalize([0.5],[0.5]), # 텐서의 픽셀 값들을 -1에서 1사이의 값으로 정규화합니다.
])

def load_image(img: str) -> torch.Tensor:
  """
  이미지를 불러와 DINOv2 모델의 입력으로 사용할 수 있는 텐서를 반환합니다.
  """

  # 1. Image.open(img): 제공된 파일 경로를 사용하여 이미지를 엽니다.
  img = Image.open(img)

  transformed_img = transform_image(img)[:3].unsqueeze(0)

  return transformed_img

In [11]:
# FAISS를 사용하여 L2 거리를 기반으로 하는 평면 인덱스(FlatL2)를 생성합니다.
def create_index(files: list) -> faiss.IndexFlatL2:
  """
  지정된 파일 목록에 있는 모든 이미지의 임베딩을 포함하는 FAISS 인덱스를 생성합니다.
  """
  index = faiss.IndexFlatL2(384)

  all_embeddings = {}

  # 모델의 기울기 계산을 비활성화합니다.
  # 이 과정은 학습이 아닌 추론단계이므로 메모리와 연산 효율을 높이기 위함입니다.
  with torch.no_grad():
    for i, file in enumerate(tqdm(files)):
      embeddings = dinov2_vits14(load_image(file).to(device))

      embedding = embeddings[0].cpu().numpy()
      all_embeddings[file] = np.array(embedding).reshape(1, -1).tolist()

      # index.add(...): FAISS 인덱스에 임베딩 벡터를 추가합니다.
      index.add(np.array(embedding).reshape(1,-1))

    with open("all_embeddings.json", "w") as f:
      f.write(json.dumps(all_embeddings))

    faiss.write_index(index, "data.bin")

    return index, all_embeddings

In [ ]:
data_index, all_embeddings = create_index(files)

In [13]:
def search_index(index: faiss.IndexFlatL2, embeddings: list, k: int = 3) -> list:
  """
  제공된 이미지와 가장 유사한 이미지들을 인덱스에서 검색합니다.
  """
  D, I = index.search(np.array(embeddings[0].reshape(1, -1)), k)

  return I[0]

In [ ]:
search_file = "COCO-128-2/valid/000000000081_jpg.rf.5262c2db56ea4568d7d32def1bde3d06.jpg"

img = cv2.resize(cv2.imread(search_file), (416, 416))

print("Input image:")

%matplotlib inline
sv.plot_image(image=img, size=(16,16))

print("*"*20)

with torch.no_grad():
  embedding = dinov2_vits14(load_image(search_file).to(device))

  indices = search_index(data_index, np.array(embedding[0].cpu()).reshape(1, -1))

  for i, index in enumerate(indices):
    print()
    print(f"Image: {i}: {files[index]}")
    img = cv2.resize(cv2.imread(files[index]), (416, 416))
    %matplotlib inline
    sv.plot_image(image=img, size=(16,16))